In [1]:
import sys
sys.path.append('../')
import time

import h5py
import epics
import numpy as np
import matplotlib.pyplot as plt

from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

Initialize cax control of siriuspy devices

In [2]:
cax = CAXCtrl()

# Scan

Parameters

In [3]:
STEP = 0.0001 # [mm]
SCALE = 1     # [um/px]
DELAY = 2     # [s]
MAXERRORCOUNT = 5

# slit1 limits
TOPMAX = 19.8
TOPMIN = 15
TOPMID = (TOPMIN + TOPMAX)/2
BOTMAX = 35.8
BOTMIN = 31
BOTMID = (BOTMIN + BOTMAX)/2

LEFTMAX = 44.56
LEFTMIN = 43.55
LEFTMID = (LEFTMIN + LEFTMAX)/2
RIGHTMAX = 45.9
RIGHTMIN = 44.88
RIGHTMID = (RIGHTMIN + RIGHTMAX)/2

Functions

In [ ]:
1e-4*1e-3*180/np.pi

5.729577951308232e-06

In [4]:
def dvfs_acquire():
    cax.dvf_A1.cmd_acquire_on()
    cax.dvf_B1.cmd_acquire_on()

def current_parameters():
    return {
        'slit1': {
            'top': cax.slit_A1.top_pos,
            'bottom': cax.slit_A1.bottom_pos,
            'left': cax.slit_A1.left_pos,
            'right': cax.slit_A1.right_pos
        },
        'dvf1': {
            'expo_time': cax.dvf_A1.acquisition_time
        },
        'dvf2': {
            'expo_time': cax.dvf_A1.acquisition_time,
            'z_pos': cax.dvf_B1.z_pos
        },
        'mirror': {
            'ry': cax.mirror.ry_pos,
            'tx': cax.mirror.tx_pos,
            'y1': cax.mirror.y1_pos,
            'y2': cax.mirror.y2_pos,
            'y3': cax.mirror.y3_pos,
            'photocollector': cax.mirror.photocurrent_signal
        }
    }

def fwhm(data):
    threshold = 0.5*np.max(data)
    mask = data > threshold
    return np.sum(mask) * SCALE

def peak(data):
    return np.max(data)

def position(data):
    return np.argmax(data) * SCALE

def move_all_slits(top, bottom, left, right):
    cax.slit_A1.move_top(value=top)
    cax.slit_A1.move_bottom(value=bottom)
    cax.slit_A1.move_left(value=left)
    cax.slit_A1.move_right(value=right)

def open_all_slits():
    move_all_slits(top=19.8,
                   bottom=35.8,
                   left=44.7,
                   right=46.1)

def move(ry_pos,delay=DELAY):
    cax.mirror.ry_pos = ry_pos
    time.sleep(delay)

def move_step(step=STEP,delay=DELAY):
    pos = cax.mirror.ry_pos + step
    move(ry_pos=pos,delay=delay)


def get_image(dvf):

    count = 0
    while count < MAXERRORCOUNT:
        try:
            if not dvf.acquisition_status:
                dvf.cmd_acquire_on()
            return dvf.image
        except Exception as err:
            print(f" WARNING. When trying to fetch image from DVF1: {err} ")
            time.sleep(2)
            count += 1
            if count < MAXERRORCOUNT:
                print("\n Repeating the procedure...\n")
            else:
                raise Exception("Client exception")



Initial state

In [5]:
local_time = time.localtime()
formatted_time = time.strftime("%Y%m%d-%H%M%S", local_time)
formatted_date = time.strftime("%Y%m%d", local_time)

formatted_time, formatted_date

('20251007-161441', '20251007')

In [6]:
ANALYSIS = 'ry'

In [7]:
parameters0 = current_parameters()

states_dir = '../states/'
state_name = f'{formatted_time}-{ANALYSIS}.txt'

with open(states_dir+state_name, 'w') as f:
    f.write(str(parameters0))

print(parameters0)

{'slit1': {'top': 17.0778125, 'bottom': 34.267812500000005, 'left': 44.153984375, 'right': 45.59546875}, 'dvf1': {'expo_time': 0.5}, 'dvf2': {'expo_time': 0.5, 'z_pos': None}, 'mirror': {'ry': 0.35131575000000004, 'tx': -0.7677535, 'y1': -0.5035784375, 'y2': 1.1945537499999999, 'y3': -0.29082031249999996, 'photocollector': -0.05132538825273514}}


Initialize scan file

In [9]:
scaname = f'scan_ry_{formatted_date}.h5'
datadir  = '/home/ids/data/'
direc    = f'{formatted_date}-Mirror-Caustic/'

Execution

In [12]:
open_all_slits()

In [10]:
file = HDF5File(filename=scaname+'_2',filedir=datadir+direc)

file.save_metadata(metadata_dict={
    'ry': cax.mirror.ry_pos
})

In [22]:
cax.mirror.ry_pos -= 3*STEP

In [25]:
ry0 = cax.mirror.ry_pos
ry0

0.35684025

In [26]:
amplitude_ry = 0.35664975 - 0.346458

In [28]:
# ry0 = 0.343029
# amplitude_ry = 0.343029 - 0.3336945
# amplitude_ry * 1e3
# positions1 = ry0 + np.linspace(0,amplitude_ry,101)[1:]
# positions2 = positions1.copy()[-2::-1]
# positions = np.hstack([positions1,positions2])

positions = np.linspace(ry0, ry0 - amplitude_ry, 51)
positions

array([0.35684025, 0.35663642, 0.35643258, 0.35622875, 0.35602491,
       0.35582108, 0.35561724, 0.35541341, 0.35520957, 0.35500574,
       0.3548019 , 0.35459807, 0.35439423, 0.3541904 , 0.35398656,
       0.35378273, 0.35357889, 0.35337506, 0.35317122, 0.35296739,
       0.35276355, 0.35255972, 0.35235588, 0.35215205, 0.35194821,
       0.35174437, 0.35154054, 0.35133671, 0.35113287, 0.35092904,
       0.3507252 , 0.35052137, 0.35031753, 0.3501137 , 0.34990986,
       0.34970603, 0.34950219, 0.34929836, 0.34909452, 0.34889069,
       0.34868685, 0.34848302, 0.34827918, 0.34807535, 0.34787151,
       0.34766768, 0.34746384, 0.34726001, 0.34705617, 0.34685234,
       0.3466485 ])

In [29]:
t0 = time.time()


for i, pos in enumerate(positions):
    print(f'{i}/{len(positions)-1}')

    #!
    #todo: deslocar feixe ate a borda. como está agora é
    #todo: a partir do valor inicial, mas deve ser a partir da borda

    move(ry_pos=pos)

    img1 = get_image(dvf=cax.dvf_A1)
    expotime1 = cax.dvf_A1.exposure_time
    img2 = get_image(dvf=cax.dvf_B1)
    expotime2 = cax.dvf_B1.exposure_time

    scan = f'scan-{i:04d}'
    scanmetadata = {
        'ry_pos': cax.mirror.ry_pos,
        'photocollector': cax.mirror.photocurrent_signal
    }
    file.save_group(grpname=scan, grpmetadata=scanmetadata)
    file.save_dataset(grpname=scan, dsetname='dvf1', 
                      dsetmetadata={'expo_time':expotime1}, dsetdata=img1)
    file.save_dataset(grpname=scan, dsetname='dvf2', 
                      dsetmetadata={'expo_time':expotime2}, dsetdata=img2)


t1 = time.time()

print()
print(f'elapsed time [min]: {(t1-t0)/60}')

0/50
1/50
2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50

elapsed time [min]: 2.271758858362834


In [30]:
f = h5py.File(name='/'.join([datadir+direc,scaname+'_2']))

fwhmsx = []
fwhmsy = []

positions = []

for scan in f:
    
    dvf2img = np.array(f[scan]['dvf2'])
    projx = np.sum(dvf2img,axis=0)
    projy = np.sum(dvf2img,axis=1)

    fwhmsx.append(fwhm(projx))
    fwhmsy.append(fwhm(projy))

    positions.append(f[scan].attrs['ry_pos'])

f.close()

In [37]:
cax.mirror.ry_pos += STEP

In [32]:
%matplotlib qt5
plt.plot(positions,fwhmsx,label='x')
plt.plot(positions,fwhmsy,label='y')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# 'top': 18.51, 'bottom': 35.7, 'left': 44.56, 'right': 46.0
rectangle_top    = 18.51
rectangle_bottom = 35.70
rectangle_left   = 44.56
rectangle_right  = 45.905
5

cax.slit_A1.top_pos    = rectangle_top
cax.slit_A1.bottom_pos = rectangle_bottom
cax.slit_A1.left_pos   = rectangle_left
cax.slit_A1.right_pos  = rectangle_right

width_y = rectangle_top   - TOPMID   + rectangle_bottom - BOTMID
width_x = rectangle_right - RIGHTMID + rectangle_left   - LEFTMID
print(f" widths = {width_x:.6f}, {width_y:.6f}")


 widths = 1.015000, 3.410000


In [204]:
# Reduce opening to 1/4 of maximum effective area.
cax.slit_A1.top_pos    = rectangle_top    - width_y * 0.42
cax.slit_A1.bottom_pos = rectangle_bottom - width_y * 0.42
#
cax.slit_A1.right_pos  = rectangle_right  - width_x * 0.30
cax.slit_A1.left_pos   = rectangle_left   - width_x * 0.40

In [96]:
topmid = cax.slit_A1.top_pos - wy/2
botmid = cax.slit_A1.bottom_pos - wy/2